In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [1]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

def start(mission: MissionBase):
    sim = CarSim(prop, mission)
    success = sim.run()
    if success:
        SimDrawer(sim).show()

c:\Users\k-ueda\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# プログラムの書き方講座2

## 1.変数って何？

- 「〇〇を見つけろ」のような命令は、見つけた結果（値）を返してくれる
  - 値を記録しておいて、後で使えるようにする為の箱のようなものを「変数」と呼ぶ
  - 箱には好きな名前をつけられる（例えばposのように）

- 下記であれば、虫を見つけた結果をposという名前で記録しておき後で利用できるようにする、という命令になる
```
pos = search(name="bug")
```

記録した値に何が入っているか？どう使うか？は関数によって異なるよ。
例えばsearchの結果の場合、pos.xと書くと「見つけた虫の前方位置」を表すといったように決まっている。

## 四則演算（足し算・引き算・掛け算・割り算）

- 数値が入っている変数や数字の間では足し算・引き算・掛け算・割り算などの計算ができる
- 足し算は`+`、引き算は`-`、掛け算は`*`、割り算は`/`で書くよ
- 例えば下記であれば、見つけた虫の前方位置の長さだけ速度0.2[m/秒]で進むという意味になる
  - 注：速度が決まっている時、何秒進めばよいかは「進みたい距離 ÷ 速度」で求まる
```
pos = search(name="bug")
move(v=0.2, t=pos.x/0.2)
```


**下のチュートリアルの説明をよく読んで、問題に取り組んでみよう。**

# チュートリアル2

ロボットを回転させて標識(標識名はsign1)が正面にくる位置で止まってから、直進して標識までのちょうど半分の距離まで進んで止まろう。
標識は開始時点でカメラの視界内に必ずあるよ。

## 取り組み方
1. 使える命令を理解する
    - move,rotate,waitはチュートリアル1と同じ。searchが追加。
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|move|一定時間だけ一定速度で前に進む|v=速度[m/秒], t=時間[秒]|move(v=0.2, t=1.0)|
|rotate|一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒]|rotate(w=90)|
|rotate|一定時間だけ一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒], t=時間[秒]|rotate(w=90, t=1.0)|
|wait|直前の命令が終わるまで待つ|-|wait()|
|search|標識を見つける（複数見つかった場合は、最も近いもの）|-|pos = search()|
|search|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = search(name="sign1")|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pos.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

![座標系](search.png)

## 注意点
- スタート時の向きはランダムに最大5度ほどずれる

In [ ]:
class Tutorial2Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((0.6, 1.75), 0.1, should_stop=True),
        ]
        self.initial_xy = (0.0, 2.0)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=1.1, y=1.6, name="sign1"),
            ]
        )


print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Tutorial2Base()).show()

最大速度 0.22 m/秒
最大回転速度 162.72 度/秒


In [3]:
class Tutorial2(Tutorial2Base):
    @staticmethod
    def command_func(alive, *, move, rotate, search, wait, **kwargs):
        ######## ここから下に「標識の方を向く」プログラムを書こう
        pos = search()
        rotate(w=90.0, t=1.0)  # 書き方の例
        wait()
        ######## ここより上に「標識の方を向く」プログラムを書こう

        ######## ここから下に「標識までの半分の距離だけ前進する」プログラムを書こう
        pos = search()
        move(v=0.2, t=3.0)  # 書き方の例
        wait()
        ######## ここより上に「標識までの半分の距離だけ前進する」プログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう


start(Tutorial2())

drive_dt=0.031, detect_dt=0.061, throttle=20
[4.087] command_func finished
[4.087] simulation_func finished
    takes 0.076s
Trajectory points : 134
Active threads : 6


100%|██████████| 46/46 [00:00<00:00, 64.08it/s] 


# チュートリアル2のヒント

- 「標識の方を向く」プログラムのヒント
    1. 「標識の方を向く」にはrotate命令を適切なwとtの値を指定し使おう。「回転したい角度」= w[度/秒] × t[秒]が成り立つようにするにはどうしたらよい？
    2. 「回転したい角度」は、車から見た標識の角度（pos.theta）と等しいことを使おう。
    3. 例えば、wをpos.thetaとするなら、tは何秒にすればいいか考えてみよう
- 「標識までの半分の距離だけ前進する」プログラムのヒント
    1. 標識までの距離はpos.xもしくはpos.rと書ける。pos.xは何m前方かを表しpos.rは標識までの距離を表す
    2. move命令のtを「進む時間」=「進みたい距離」/「速度」となるように設定しよう